In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r'C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\phishing_total.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44519 entries, 0 to 44518
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   file_name      44519 non-null  object
 1   phishing_type  44519 non-null  object
 2   speaker        44519 non-null  int64 
 3   text           44519 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.4+ MB


In [7]:
df.head()

,file_name,phishing_type,speaker,text
0,phishing_000,가족지인사칭형,0,"여보세요, OOO야. 나 아빠야."
1,phishing_000,가족지인사칭형,1,아빠? 목소리가 좀 이상한데요.
2,phishing_000,가족지인사칭형,0,아빠 핸드폰이 고장 나서 친구 폰으로 걸었어. 급하게 돈이 필요해서.
3,phishing_000,가족지인사칭형,1,무슨 일인데 그렇게 급해요?
4,phishing_000,가족지인사칭형,0,교통사고가 나서 병원비가 필요해. 믿고 바로 보내줄 수 있지?


In [4]:
df['phishing_type'].unique()

array(['가족지인사칭형', '세금환급형', '콜백스미싱형', '투자권유형', '대출빙자형', '메신저피싱형', '택배사칭형',
       '기관사칭형'], dtype=object)

In [8]:
# 원하는 피싱 유형 리스트
phishing_types_to_count = ['콜백스미싱형', '세금환급형', '투자권유형']

# 각 유형별 file_name의 고유 개수 세기
counts = df[df['phishing_type'].isin(phishing_types_to_count)] \
            .groupby('phishing_type')['file_name'].nunique()

print(counts)


phishing_type
세금환급형     103
콜백스미싱형    108
투자권유형      91
Name: file_name, dtype: int64


In [ ]:
import os
import time
import re
import pandas as pd
import openai

# 한글 문장만 추출하는 함수
def extract_korean(text):
    """
    입력된 텍스트에서 한글, 숫자, 일부 기호만 남기고 나머지는 삭제하여 반환합니다.
    여러 줄이 있을 경우 쉼표로 합칩니다.
    """
    korean_lines = []
    for line in text.split('\n'):
        # 한글이 포함된 줄만 추출
        if re.search(r'[가-힣]', line):
            # 양쪽 공백 및 따옴표 제거
            clean = line.strip().strip("'\"“”")
            # 영어, 괄호 등 제거 (필요시 추가)
            clean = re.sub(r'[a-zA-Z]+.*', '', clean)
            # 한글, 숫자, 기호만 남기기
            clean = re.sub(r'[^가-힣0-9\s,\.?!~…·:;\'\"()-]', '', clean)
            if clean and len(clean) > 1:
                korean_lines.append(clean)
    return ', '.join(korean_lines)

# OpenAI GPT를 이용한 역번역(Back-translation) 함수
def gpt_back_translate_en(text, client, model="gpt-4o-mini"):
    """
    입력된 한국어 문장을 영어로 번역 후 다시 한국어로 자연스럽게 재번역합니다.
    최종적으로 한글 문장만 반환합니다.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "너는 한국 보이스피싱 탐지 시스템 개발 프로젝트에 참여 중인 데이터 증강 전문가야. "
                        "사용자가 주는 한국어 문장은 보이스피싱 사례 텍스트야. "
                        "너의 임무는 이 문장을 먼저 영어로 자연스럽게 번역한 뒤, 다시 한국어로 자연스럽고 다양하게 재구성해주는 거야. "
                        "단, 원래 문장의 의미와 맥락은 유지하되 표현을 바꾸고, 자연스럽고 실제 통화처럼 들리도록 만들어야 해. "
                        "최종 출력은 번역된 한국어 문장만 보여줘."
                    ),
                },
                {
                    "role": "user",
                    "content": f"다음 문장을 영어로 번역한 후 다시 한국어로 자연스럽게 재번역해줘. 최종출력은 한국어 문장만:\n\n'{text}'",
                },
            ],
            temperature=1.0,
            max_tokens=500,
        )
        result = response.choices[0].message.content.strip()
        # 한글 문장만 추출
        return extract_korean(result)
    except Exception as e:
        print(f"❌ 역번역 중 오류 발생: {e}")
        return None

# 여러 문장(배치)을 한 번에 증강하는 함수
def batch_back_translate(texts, client, model="gpt-4o-mini", wait_sec=0.5):
    """
    여러 문장을 받아 GPT 역번역을 순차적으로 수행하고, 결과 리스트를 반환합니다.
    각 요청 사이에 wait_sec(초) 만큼 대기합니다.
    """
    results = []
    for text in texts:
        result = gpt_back_translate_en(text, client, model)
        # 오류 발생 시 원본 텍스트를 그대로 사용
        results.append(result if result is not None else text)
        time.sleep(wait_sec)
    return results

# 데이터프레임을 phishing_type별, 텍스트 길이 기준으로 증강 및 실시간 저장
def augment_data_by_text_length_realtime_save(
    df, phishing_types, client, save_path='augmented_data.csv', batch_size=10, model="gpt-4o-mini"
):
    """
    phishing_types: {'피싱유형명': 생성할 대화(파일) 수, ...} 형태의 dict
    각 유형별로 텍스트 길이가 긴 대화(파일)부터 target_count개씩 증강하여 실시간 저장
    """
    # 저장할 컬럼명 정의 및 파일 초기화
    columns = ['file_name', 'phishing_type', 'speaker', 'text']
    pd.DataFrame(columns=columns).to_csv(save_path, index=False, encoding='utf-8-sig')
    # 각 피싱 유형별로 증강 반복
    for phishing_type, target_count in phishing_types.items():
        type_data = df[df['phishing_type'] == phishing_type].copy()
        if type_data.empty:
            continue
        # 각 파일별 전체 텍스트 길이 계산
        file_text_lengths = {}
        for file_name in type_data['file_name'].unique():
            file_data = type_data[type_data['file_name'] == file_name]
            total_text = ' '.join(file_data['text'].astype(str))
            file_text_lengths[file_name] = len(total_text)
        # 텍스트 길이 기준 내림차순 정렬
        sorted_files = sorted(file_text_lengths.items(), key=lambda x: x[1], reverse=True)
        count = 0
        # target_count만큼 긴 파일부터 증강
        for file_name, text_length in sorted_files:
            if count >= target_count:
                break
            file_data = type_data[type_data['file_name'] == file_name]
            augmented_file_name = f"{file_name}_augmented_{count+1}"
            rows = list(file_data.iterrows())
            # 배치 단위로 증강
            for batch_start in range(0, len(rows), batch_size):
                batch_rows = rows[batch_start:batch_start+batch_size]
                texts = [row['text'] for _, row in batch_rows]
                augmented_texts = batch_back_translate(texts, client, model)
                for ((_, row), aug_text) in zip(batch_rows, augmented_texts):
                    # 결과가 비어있지 않은 경우만 저장
                    if aug_text and len(aug_text) > 1:
                        new_row = {
                            'file_name': augmented_file_name,
                            'phishing_type': row['phishing_type'],
                            'speaker': row['speaker'],
                            'text': aug_text
                        }
                        # 증강 결과를 실시간으로 파일에 append 저장
                        pd.DataFrame([new_row]).to_csv(save_path, mode='a', header=False, index=False, encoding='utf-8-sig')
            count += 1

# 메인 실행부: 데이터 로드 및 증강 함수 실행
if __name__ == "__main__":
    # 원본 피싱 데이터 로드
    df = pd.read_csv(r'C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\phishing_total.csv')
    # OpenAI API 클라이언트 생성 (환경변수에서 API 키 로드)
    client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    # 증강할 피싱 유형 및 생성할 대화(파일) 수 지정
    phishing_types_to_augment = {
        '콜백스미싱형': 71,
        '세금환급형': 15,
        '투자권유형': 29
    }
    # 증강 함수 실행 (실시간 저장)
    augment_data_by_text_length_realtime_save(
        df, phishing_types_to_augment, client, batch_size=10, model="gpt-4o-mini"
    )


In [39]:
import os
import time
import re
import pandas as pd
import openai
import random

# 한글 문장만 추출 함수
def extract_korean(text):
    """
    입력된 텍스트에서 한글, 숫자, 일부 기호만 남기고 나머지는 삭제하여 반환합니다.
    여러 줄이 있을 경우 쉼표로 합칩니다.
    """
    korean_lines = []
    for line in text.split('\n'):
        if re.search(r'[가-힣]', line):
            clean = line.strip().strip("'\"“”")
            clean = re.sub(r'[a-zA-Z]+.*', '', clean)
            clean = re.sub(r'[^가-힣0-9\s,\.?!~…·:;\'\"()-]', '', clean)
            if clean and len(clean) > 1:
                korean_lines.append(clean)
    return ', '.join(korean_lines)

# 역번역 함수
def gpt_back_translate_de(text, client, model="gpt-4.1-mini"):
    """
    입력된 한국어 문장을 독일어로 번역 후 다시 한국어로 자연스럽게 재구성합니다.
    """
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "너는 한국 보이스피싱 탐지 시스템 개발 프로젝트 데이터 증강 전문가야. "
                        "한국어 문장을 먼저 독일어로 번역 후 다시 한국어로 자연스럽게 재구성해줘. "
                        "의미와 맥락을 유지하되 실제 통화처럼 들리도록 표현을 다양화해줘. "
                        "최종 출력은 한국어 문장만."
                    )
                },
                {
                    "role": "user",
                    "content": f"문장을 독일어로 번역한 후 다시 한국어로 재번역해줘:\n\n'{text}'"
                }
            ],
            temperature=0.9,
            max_tokens=500,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"역번역 오류: {e}")
        return None

# clean_backtranslate_output: 불필요 접두어 및 레이블 제거
def clean_backtranslate_output(raw):
    """
    GPT 결과에서 '독일어 번역:', '한국어 재번역:' 등 불필요 레이블과 빈 문자열을 제거합니다.
    """
    parts = re.split(r'독일어 번역:.*?한국어 재번역:|독일어 번역:|한국어 재번역:', raw)
    cleaned = [p.strip().strip('",') for p in parts if p and p.strip().strip('",')]
    return ', '.join(cleaned)

# 배치 역번역 수행 함수
def batch_back_translate(texts, client, model="gpt-4.1-mini", wait_sec=0.5):
    results = []
    for t in texts:
        raw = gpt_back_translate_de(t, client, model)
        if raw:
            cleaned = clean_backtranslate_output(raw)
            results.append(extract_korean(cleaned))
        else:
            results.append(t)
        time.sleep(wait_sec)
    return results

# 길이 구간별 대상 파일 선택 및 부족분 보충 함수
def select_and_fill(file_groups, length_range, count, pool):
    lo, hi = length_range
    candidates = file_groups[
        (file_groups['text_len'] >= lo) &
        (file_groups['text_len'] <= hi)
    ]['file_name'].tolist()
    selected = []
    if candidates:
        take = min(len(candidates), count)
        selected = random.sample(candidates, take)
    deficit = count - len(selected)
    if deficit > 0 and pool:
        fill = [f for f in pool if f not in selected]
        if len(fill) <= deficit:
            selected += fill
        else:
            selected += random.sample(fill, deficit)
    return selected

# 기관사칭형 데이터 500건 증강 메인 함수
def augment_gigansachingtype_data(df, client, save_path='augmented_de_500.csv', batch_size=10):
    df_type = df[df['phishing_type'] == '기관사칭형'].copy()
    if df_type.empty:
        print("기관사칭형 데이터가 없습니다.")
        return

    # 파일별 전체 텍스트 결합 및 길이 계산
    file_groups = df_type.groupby('file_name')['text'] \
        .apply(lambda ts: ' '.join(ts)).reset_index()
    file_groups['text_len'] = file_groups['text'].str.len()

    all_files = file_groups['file_name'].tolist()

    # 길이 구간별 목표 건수
    buckets = [
        ((100, 200), 138),
        ((201, 400), 220),
        ((401, 600), 110),
        ((601, 800), 55),
        ((801,1000), 22),
        ((1001, float('inf')), 6),
    ]

    target_files = []
    remaining_pool = set(all_files)
    for length_range, cnt in buckets:
        sel = select_and_fill(file_groups, length_range, cnt, list(remaining_pool))
        target_files.extend(sel)
        remaining_pool -= set(sel)

    # 중복 제거 후 총 500개 맞추기
    target_files = list(dict.fromkeys(target_files))
    if len(target_files) < 500:
        need = 500 - len(target_files)
        extras = list(remaining_pool)
        take = min(len(extras), need)
        target_files.extend(random.sample(extras, take))
    elif len(target_files) > 500:
        target_files = random.sample(target_files, 500)

    print(f"총 증강 대상 파일 수: {len(target_files)}")

    # 결과 CSV 초기화
    cols = ['file_name', 'phishing_type', 'speaker', 'text']
    pd.DataFrame(columns=cols).to_csv(save_path, index=False, encoding='utf-8-sig')

    # 파일 단위로 배치 증강 및 저장
    for idx, fname in enumerate(target_files, 1):
        print(f"[{idx}/500] 증강 중: {fname}")
        rows = df_type[df_type['file_name'] == fname].reset_index(drop=True)
        aug_fname = f"{fname}_de_augmented_{idx}"
        for i in range(0, len(rows), batch_size):
            batch = rows.iloc[i:i+batch_size]
            aug_texts = batch_back_translate(batch['text'].tolist(), client)
            for orig, aug in zip(batch.itertuples(), aug_texts):
                if aug and len(aug) > 1:
                    pd.DataFrame([{
                        'file_name': aug_fname,
                        'phishing_type': orig.phishing_type,
                        'speaker': orig.speaker,
                        'text': aug
                    }]).to_csv(save_path, mode='a', header=False, index=False, encoding='utf-8-sig')
    print("✅ 500건 증강 완료:", save_path)

if __name__ == "__main__":
    # 데이터 로드
    df = pd.read_csv(r'C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset_create\Yongjae\기관사칭형_en_dataset.csv', encoding='utf-8-sig')
    # OpenAI 클라이언트 생성
    client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    # 증강 실행
    augment_gigansachingtype_data(df, client, save_path='augmented_gigansachingtype_de_500.csv', batch_size=10)


총 증강 대상 파일 수: 500
[1/500] 증강 중: phishing_1930_en
[2/500] 증강 중: phishing_1024_en
[3/500] 증강 중: phishing_1873_en
[4/500] 증강 중: phishing_1514_en
[5/500] 증강 중: phishing_1311_en
[6/500] 증강 중: phishing_758_en
[7/500] 증강 중: phishing_1399_en
[8/500] 증강 중: phishing_1677_en
[9/500] 증강 중: phishing_930_en
[10/500] 증강 중: phishing_1207_en
[11/500] 증강 중: phishing_1412_en
[12/500] 증강 중: phishing_1557_en
[13/500] 증강 중: phishing_1117_en
[14/500] 증강 중: phishing_1848_en
[15/500] 증강 중: phishing_925_en
[16/500] 증강 중: phishing_1193_en
[17/500] 증강 중: phishing_1637_en
[18/500] 증강 중: phishing_911_en
[19/500] 증강 중: phishing_1212_en
[20/500] 증강 중: phishing_1120_en
[21/500] 증강 중: phishing_1197_en
[22/500] 증강 중: phishing_1945_en
[23/500] 증강 중: phishing_728_en
[24/500] 증강 중: phishing_1016_en
[25/500] 증강 중: phishing_1490_en
[26/500] 증강 중: phishing_986_en
[27/500] 증강 중: phishing_1691_en
[28/500] 증강 중: phishing_1690_en
[29/500] 증강 중: phishing_1885_en
[30/500] 증강 중: phishing_1322_en
[31/500] 증강 중: phishing_1574_en
[32/5

In [ ]:
import pandas as pd
import openai
import os
import time

class GPT4oMiniTranslator:
    """
    OpenAI GPT-4o-mini를 활용한 번역 및 역번역(Back-translation) 데이터 증강 클래스
    """
    def __init__(self, api_key=None, model='gpt-4o-mini'):
        # API 키는 인자로 받거나 환경변수에서 불러옴
        self.api_key = api_key or os.getenv('OPENAI_API_KEY')
        self.model = model
        # OpenAI 클라이언트 생성
        self.client = openai.OpenAI(api_key=self.api_key)

    def batch_translate(self, texts, source_lang='ko', target_lang='en', max_retries=5, wait_sec=5):
        """
        여러 문장을 한 번에 번역(또는 표현 다양화)하여 리스트로 반환
        - texts: 번역할 문장 리스트
        - source_lang: 원본 언어 코드
        - target_lang: 번역 언어 코드
        - max_retries: 실패 시 최대 재시도 횟수
        - wait_sec: 실패 시 대기 시간(초)
        """
        # 번역 목적에 맞는 프롬프트 작성
        if target_lang == 'en':
            prompt = (
                "너는 전문 번역가이자 데이터 증강 전문가야. 아래의 문장들을 데이터 증강을 위해 의미는 유지하되, "
                "표현을 다양하게 바꿔서 영어로 번역해. 각 문장은 번호로 구분되어 있어. "
                "아무 설명도 하지 말고, 오직 번역 결과만 출력해. 각 번역 결과도 번호로 구분해서 출력해.\n\n"
            )
            prompt += "\n".join([f"{i+1}. {t}" for i, t in enumerate(texts)])
        elif target_lang == 'ko':
            prompt = (
                "너는 전문 번역가이자 데이터 증강 전문가야. 아래의 문장들을 데이터 증강을 위해 의미는 유지하되, "
                "표현을 다양하게 바꿔서 한국어로 번역해. 각 문장은 번호로 구분되어 있어. "
                "아무 설명도 하지 말고, 오직 번역 결과만 출력해. 출력에 영어가 섞이면 안 돼. "
                "각 번역 결과도 번호로 구분해서 출력해.\n\n"
            )
            prompt += "\n".join([f"{i+1}. {t}" for i, t in enumerate(texts)])
        else:
            prompt = (
                f"너는 전문 번역가이자 데이터 증강 전문가야. 아래의 문장들을 데이터 증강을 위해 의미는 유지하되, "
                f"표현을 다양하게 바꿔서 {target_lang}로 번역해. 각 문장은 번호로 구분되어 있어. "
                "아무 설명도 하지 말고, 오직 번역 결과만 출력해. 각 번역 결과도 번호로 구분해서 출력해.\n\n"
            )
            prompt += "\n".join([f"{i+1}. {t}" for i, t in enumerate(texts)])

        # API 호출 및 예외/재시도 처리
        for attempt in range(max_retries):
            try:
                response = self.client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": "너는 전문 번역가이자 데이터 증강 전문가야."},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0.3
                )
                result = response.choices[0].message.content.strip()
                # 번호별로 결과 파싱
                outputs = []
                for i in range(len(texts)):
                    prefix = f"{i+1}."
                    if prefix in result:
                        next_prefix = f"{i+2}."
                        start = result.find(prefix) + len(prefix)
                        end = result.find(next_prefix) if next_prefix in result else None
                        outputs.append(result[start:end].strip())
                    else:
                        outputs.append("")
                return outputs
            except openai.RateLimitError:
                print(f"429 오류: {wait_sec}초 후 재시도... (시도 {attempt+1}/{max_retries})")
                time.sleep(wait_sec)
            except Exception as e:
                print(f"번역 중 오류 발생: {e}")
                time.sleep(wait_sec)
        print("최대 재시도 횟수 초과, 원본 텍스트 반환")
        return texts

    def batch_back_translate(self, texts, intermediate_lang='en'):
        """
        번역(ko→en) 후 다시 역번역(en→ko)하여 데이터 증강
        - texts: 원본 한글 문장 리스트
        - intermediate_lang: 중간 언어(기본 'en')
        """
        en_list = self.batch_translate(texts, source_lang='ko', target_lang=intermediate_lang)
        time.sleep(0.5)
        ko_list = self.batch_translate(en_list, source_lang=intermediate_lang, target_lang='ko')
        time.sleep(0.5)
        return ko_list

def augment_data_by_text_length_realtime_save(df, translator, save_path='augmented_data.csv', batch_size=10):
    """
    file_name별로 텍스트 길이가 가장 긴 파일을 골라 증강(역번역)하고, 결과를 실시간으로 저장
    - df: 원본 데이터프레임
    - translator: GPT4oMiniTranslator 인스턴스
    - save_path: 증강 데이터 저장 경로
    - batch_size: 한 번에 처리할 문장 수
    """
    augmented_data = []
    columns = list(df.columns)
    # 결과 파일 초기화(헤더만)
    pd.DataFrame(columns=columns).to_csv(save_path, index=False, encoding='utf-8-sig')
    # file_name이 'phishing_1670'인 것만 처리
    type_data = df[df['file_name'] == 'phishing_1670'].copy()
    if type_data.empty:
        print("phishing_1670 데이터가 없습니다.")
        return df
    # 파일별 전체 텍스트 길이 계산
    file_text_lengths = {}
    for file_name in type_data['file_name'].unique():
        file_data = type_data[type_data['file_name'] == file_name]
        total_text = ' '.join(file_data['text'].astype(str))
        file_text_lengths[file_name] = len(total_text)
    # 텍스트 길이 기준 내림차순 정렬
    sorted_files = sorted(file_text_lengths.items(), key=lambda x: x[1], reverse=True)
    print(f"길이 순 정렬 결과:")
    for i, (file_name, length) in enumerate(sorted_files):
        print(f"  {i+1}. {file_name}: {length}자")
    count = 0
    target_count = 1  # 'phishing_1670'만 증강
    for file_name, text_length in sorted_files:
        if count >= target_count:
            break
        file_data = type_data[type_data['file_name'] == file_name]
        augmented_file_name = f"{file_name}_augmented_{count+1}"
        print(f"  처리 중: {file_name} ({text_length}자) -> {augmented_file_name}")
        rows = list(file_data.iterrows())
        # 배치 단위로 증강
        for batch_start in range(0, len(rows), batch_size):
            batch_rows = rows[batch_start:batch_start+batch_size]
            texts = [row['text'] for _, row in batch_rows]
            # 역번역 증강
            augmented_texts = translator.batch_back_translate(texts)
            for ((_, row), aug_text) in zip(batch_rows, augmented_texts):
                augmented_row = row.copy()
                augmented_row['file_name'] = augmented_file_name
                augmented_row['text'] = aug_text
                # 증강 결과를 실시간으로 파일에 append 저장
                pd.DataFrame([augmented_row]).to_csv(save_path, mode='a', header=False, index=False, encoding='utf-8-sig')
                augmented_data.append(augmented_row)
                time.sleep(0.2)
        count += 1
        print(f"  완료: {file_name} - {count}/{target_count}")
    # 증강 결과를 DataFrame으로 반환(필요시)
    if augmented_data:
        augmented_df = pd.DataFrame(augmented_data)
        return pd.concat([df, augmented_df], ignore_index=True)
    return df

if __name__ == "__main__":
    # 원본 데이터 로드
    df = pd.read_csv(r'C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\phishing_total.csv')
    # file_name이 'phishing_1670'인 것만 남김
    df = df[df['file_name'] == 'phishing_1670'].copy()
    # 번역기 인스턴스 생성
    translator = GPT4oMiniTranslator()
    print("phishing_1670 데이터 증강을 시작합니다...")
    print(f"원본 데이터 크기: {len(df)}")
    # 증강 함수 실행
    augmented_df = augment_data_by_text_length_realtime_save(df, translator, batch_size=10)
    print(f"\n증강 완료!")
    print(f"최종 데이터 크기: {len(augmented_df)}")
    print(f"증강된 데이터 개수: {len(augmented_df) - len(df)}")
    print("증강된 데이터가 'augmented_data.csv'로 실시간 저장되었습니다.")


phishing_1670 데이터 증강을 시작합니다...
원본 데이터 크기: 26
길이 순 정렬 결과:
  1. phishing_1670: 1032자
  처리 중: phishing_1670 (1032자) -> phishing_1670_augmented_1
  완료: phishing_1670 - 1/1

증강 완료!
최종 데이터 크기: 52
증강된 데이터 개수: 26
증강된 데이터가 'augmented_data.csv'로 실시간 저장되었습니다.


In [ ]:
import pandas as pd
import os
import time
import re
from dotenv import load_dotenv
from openai import OpenAI

# 1. 환경 변수에서 OpenAI API 키 불러오기 (.env 파일 필요)
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    raise ValueError("❌ .env에 OPENAI_API_KEY가 설정되지 않았습니다.")

# 2. OpenAI API 클라이언트 생성
client = OpenAI(api_key=openai_api_key)

# 3. 역번역 함수 정의 (GPT 응답 줄 수 일치 확인 + 로그 저장)
def back_translate_with_openai(speaker_list, text_list, file_name):
    """
    speaker_list, text_list: 각 대화의 화자와 발화 리스트
    file_name: 현재 처리 중인 대화 파일명(로그 저장용)
    - 각 발화를 SPEAKER 태그로 묶어 GPT에 전달
    - 영어로 번역 후 다시 한국어로 번역하도록 프롬프트 설계
    - 응답 줄 수가 원본과 다르면 실패 처리 및 로그 저장
    - 정상적으로 파싱된 경우, 각 발화별 역번역 결과 리스트 반환
    """
    tagged_lines = [f"SPEAKER_{spk}: {txt}" for spk, txt in zip(speaker_list, text_list)]
    joined_text = "\n".join(tagged_lines)

    prompt = f"""
너는 전문 번역가이자 데이터 증강 전문가야.
아래는 한국어로 된 보이스피싱 대화야. 데이터 증강을 위해 각 문장을 영어로 번역한 뒤, 다시 한국어로 번역해.
단, 의미는 최대한 비슷하게 유지하되 표현은 다양하게 바꿔주고, 유의어를 적극적으로 활용해.
각 문장은 반드시 SPEAKER_0: 또는 SPEAKER_1: 으로 시작해야 해.

다음 지침을 엄격히 따르세요.
1. 각 발화를 영어로 번역한 뒤, 다시 한국어로 번역.
2. 의미는 최대한 비슷하게 유지하되, 표현은 다양하게 바꿔서 써줘. 유의어 치환, 어순 변경 등 적극적으로 활용.
3. **출력 형식은 반드시 SPEAKER 태그 + 원문과 동일한 줄 수(총 {len(text_list)}줄)를 유지.**
4. 줄 수가 다를 경우 작업은 실패. 
5. **최종 출력은 반드시 한국어로만 출력합니다.**

입력:
{joined_text}

역번역 한국어 결과:
""".strip()

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.9
        )
        reply = response.choices[0].message.content

        # GPT 응답 원본 로그 저장 (디버깅/재현용)
        with open("gpt_raw_log.txt", "a", encoding="utf-8") as log_file:
            log_file.write(f"\n\n===== {file_name} =====\n{reply}\n")

        # SPEAKER 태그로 분리 및 파싱
        matches = re.findall(r"(?i)SPEAKER[_ ]?(\d):\s*(.+)", reply)
        if len(matches) != len(text_list):
            print(f"[경고] 문장 수 불일치 - 원문: {len(text_list)}, GPT 응답: {len(matches)}")
            # 실패 파일명 기록
            with open("failed_files.txt", "a", encoding="utf-8") as fail_log:
                fail_log.write(file_name + "\n")
            return [None] * len(text_list)

        return [text for _, text in matches]

    except Exception as e:
        print(f"[OpenAI API 오류] {e}")
        with open("failed_files.txt", "a", encoding="utf-8") as fail_log:
            fail_log.write(file_name + "\n")
        return [None] * len(text_list)

# 4. 주요 설정값
INPUT_FILE = r"C:\Users\user\Desktop\woogawooga\woogawooga_project\dataset\phishing_total.csv"
OUTPUT_FILE = "cy_jgang.csv"
TARGET_TYPES = ['콜백스미싱형', '세금환급형', '투자권유형']

# 5. 이미 처리한 file_name 목록 불러오기 (중복 증강 방지)
def load_processed_file_names():
    if not os.path.exists(OUTPUT_FILE):
        return set()
    try:
        processed_df = pd.read_csv(OUTPUT_FILE)
        return set(processed_df['file_name'].unique())
    except:
        return set()

# 6. 데이터 불러오기 및 필터링
df = pd.read_csv(INPUT_FILE)
df = df[df['phishing_type'].isin(TARGET_TYPES)]
grouped = df.groupby("file_name")
processed_files = load_processed_file_names()

# 7. 증강 결과 CSV 실시간 저장 (중간중간 저장, 중단시 재시작 가능)
with open(OUTPUT_FILE, 'a', encoding='utf-8', newline='') as f_out:
    # 파일이 비어있으면 헤더 작성
    if os.path.getsize(OUTPUT_FILE) == 0:
        f_out.write("file_name,speaker,phishing_type,text,back_translated\n")

    # file_name(대화 단위)별로 반복 처리
    for file_name, group in grouped:
        if file_name in processed_files:
            continue  # 이미 처리된 대화는 건너뜀

        speaker_list = group['speaker'].tolist()
        text_list = group['text'].tolist()
        phishing_type = group['phishing_type'].iloc[0]

        # 역번역 수행
        backtranslated_list = back_translate_with_openai(speaker_list, text_list, file_name)

        # 결과 저장 (실시간 append)
        for spk, orig, back in zip(speaker_list, text_list, backtranslated_list):
            orig_safe = str(orig).replace("\n", " ").replace(",", " ")
            back_safe = str(back or "").replace("\n", " ").replace(",", " ")
            f_out.write(f"{file_name},{spk},{phishing_type},{orig_safe},{back_safe}\n")

        print(f" 처리 완료: {file_name}")
        time.sleep(1.5)  # OpenAI API rate limit 대응용 딜레이


 처리 완료: phishing_100
 처리 완료: phishing_101
 처리 완료: phishing_102
 처리 완료: phishing_103
 처리 완료: phishing_104
 처리 완료: phishing_105
 처리 완료: phishing_106
 처리 완료: phishing_107
 처리 완료: phishing_108
 처리 완료: phishing_109


KeyboardInterrupt: 